In [1]:
# Memuat API key dari file .env
from dotenv import load_dotenv

load_dotenv()

True

## No memory

In [2]:
# Menginisialisasi agent biasa tanpa checkpointer (stateless / tanpa memori)
from langchain.agents import create_agent


agent = create_agent(
    "google_genai:gemini-3.1-flash-lite"
)

In [3]:
# Mengirim pesan perkenalan diri dan preferensi warna
from langchain.messages import HumanMessage

question = HumanMessage(content="Halo nama saya Dimas dan warna kesukaanku adalah hijau")

response = agent.invoke(
    {"messages": [question]} 
)

In [4]:
# Menampilkan respons awal agent
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Halo nama saya Dimas dan warna kesukaanku adalah hijau', additional_kwargs={}, response_metadata={}, id='a2397685-87aa-4521-9869-efeee9f5da9d'),
              AIMessage(content=[{'type': 'text', 'text': 'Halo Dimas! Salam kenal ya. Wah, warna hijau itu pilihan yang bagus sekali. Hijau sering kali memberikan kesan yang segar, tenang, dan alami.\n\nApa nih yang membuat kamu suka warna hijau? Apakah karena suka suasana alam atau ada alasan lain?', 'extras': {'signature': 'EjQKMgERTTIPMAVkvh8LIFaQQyC6OCL2wsVKNreT4w0rFbVJKK7K0bRVY4cAyGadvuYL/OrC'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f68f0-7cbd-78d0-9c1d-5662b1fb39d4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 53, 'total_tokens': 66, 'input_token_details': {'cache_read': 0}})]}


In [5]:
# Mengirim pertanyaan tindak lanjut untuk menguji ingatan agent
# Tanpa memori, agent tidak akan tahu apa warna kesukaan pengguna dari pesan sebelumnya
question = HumanMessage(content="Apa warna kesukaanku?")

response = agent.invoke(
    {"messages": [question]} 
)

pprint(response)

{'messages': [HumanMessage(content='Apa warna kesukaanku?', additional_kwargs={}, response_metadata={}, id='cf12be85-5f04-4296-b707-c48dbc67fcd1'),
              AIMessage(content=[{'type': 'text', 'text': 'Sebagai model kecerdasan buatan, saya tidak memiliki akses ke memori pribadi atau kehidupan Anda, jadi saya belum tahu apa warna kesukaan Anda.\n\nNamun, kalau Anda mau memberi tahu saya, saya akan ingat! Jadi, **apa warna kesukaan Anda?**', 'extras': {'signature': 'EjQKMgERTTIPqo9YIp30N5pq1Joyj0dEskF1oCfaCpRSMWYZUDuc8syoTRM7YMgTOph7ZZrS'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f68f0-d441-76f0-9852-c8428aa00284-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 54, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}})]}


## Memory

In [6]:
# Menginisialisasi agent dengan checkpointer InMemorySaver untuk menyimpan state percakapan (stateful / memiliki memori)
from langgraph.checkpoint.memory import InMemorySaver  


agent = create_agent(
    "google_genai:gemini-3.1-flash-lite",
    checkpointer=InMemorySaver(),  
)

In [7]:
# Mengirim pesan perkenalan dengan menyertakan konfigurasi thread_id
# thread_id digunakan untuk menandai/mengelompokkan sesi percakapan unik
from langchain.messages import HumanMessage

question = HumanMessage(content="Halo nama saya Dimas dan warna kesukaanku adalah hijau")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [9]:
# Menampilkan respons agent
pprint(response)

{'messages': [HumanMessage(content='Halo nama saya Dimas dan warna kesukaanku adalah hijau', additional_kwargs={}, response_metadata={}, id='ab5a23ac-def0-4f1a-a634-07a2fd4d08eb'),
              AIMessage(content=[{'type': 'text', 'text': 'Halo Dimas! Salam kenal ya. Wah, warna hijau itu pilihan yang bagus, kesannya segar dan menenangkan, seperti alam atau dedaunan.\n\nAda alasan khusus kenapa kamu suka warna hijau, Dimas? Atau mungkin ada hal lain yang ingin kamu ceritakan atau tanyakan hari ini?', 'extras': {'signature': 'EjQKMgERTTIPpJL5UAtLQKT7bOqipc/3KrsoBzDZGA8vUUtoZP8zABHAUTyKcZblo7sVVB4l'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f68f1-c878-7453-a22f-065403795ef3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 61, 'total_tokens': 74, 'input_token_details': {'cache_read': 0}})]}


In [10]:
# Mengirim pertanyaan tindak lanjut menggunakan thread_id yang sama
# Dengan checkpointer dan thread_id yang sama, agent dapat mengingat pesan sebelumnya dan menjawab dengan tepat
question = HumanMessage(content="Apa warna kesukaanku?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

pprint(response)

{'messages': [HumanMessage(content='Halo nama saya Dimas dan warna kesukaanku adalah hijau', additional_kwargs={}, response_metadata={}, id='ab5a23ac-def0-4f1a-a634-07a2fd4d08eb'),
              AIMessage(content=[{'type': 'text', 'text': 'Halo Dimas! Salam kenal ya. Wah, warna hijau itu pilihan yang bagus, kesannya segar dan menenangkan, seperti alam atau dedaunan.\n\nAda alasan khusus kenapa kamu suka warna hijau, Dimas? Atau mungkin ada hal lain yang ingin kamu ceritakan atau tanyakan hari ini?', 'extras': {'signature': 'EjQKMgERTTIPpJL5UAtLQKT7bOqipc/3KrsoBzDZGA8vUUtoZP8zABHAUTyKcZblo7sVVB4l'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f68f1-c878-7453-a22f-065403795ef3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 61, 'total_tokens': 74, 'input_token_details': {'cache_read': 0}}),
              Huma